In [ ]:
# 1. Clone code từ GitHub (Runtime -> Change runtime type -> GPU)
#    Repo private: dùng "https://<GITHUB_TOKEN>@github.com/<USERNAME>/engagement-v5.git"
REPO_URL = "https://github.com/<USERNAME>/engagement-v5.git"

!rm -rf /content/engagement-v5
!git clone -q {REPO_URL} /content/engagement-v5
%cd /content/engagement-v5
!pip install -q -r requirements.txt kagglehub

In [ ]:
# 2. Chuẩn bị dữ liệu - chọn MỘT trong hai cách
USE_KAGGLEHUB = True

if USE_KAGGLEHUB:
    # Cách A: tải thẳng từ Kaggle. Thêm KAGGLE_USERNAME và KAGGLE_KEY vào Colab Secrets (biểu tượng chìa khoá).
    import os
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    !python scripts/download_kaggle_data.py
    CONFIG = "configs/colab_auto.yaml"
else:
    # Cách B: dữ liệu đã có trong Google Drive -> sửa đường dẫn trong configs/colab.yaml
    from google.colab import drive
    drive.mount("/content/drive")
    CONFIG = "configs/colab.yaml"

In [ ]:
# 3. Kiểm tra dữ liệu
!python scripts/diagnose.py --config {CONFIG}

In [ ]:
# 4. Chạy thử nhanh
!python scripts/train.py --config {CONFIG} --set DRY_RUN=true CHECKPOINT_DIR=/content/dry_run

In [ ]:
# 5. Train đầy đủ + đánh giá test
!python scripts/train.py --config {CONFIG}

In [ ]:
# 6. Xem kết quả (thư mục = CHECKPOINT_DIR trong config)
import json
from IPython.display import Image, display
import sys; sys.path.insert(0, "src")
from engagement.config import load_config

OUT_DIR = load_config(CONFIG)["CHECKPOINT_DIR"]
print(json.dumps(json.load(open(f"{OUT_DIR}/test_results.json")), indent=2, ensure_ascii=False))
display(Image(f"{OUT_DIR}/confusion_matrix.png"))
display(Image(f"{OUT_DIR}/f1_coverage.png"))